In [28]:
%pip install kete
import kete
import matplotlib.pyplot as plt
import numpy as np

import pandas as pd
import numpy as np


# Load ephemeris data en masse
df = pd.read_csv('cleaned_data/atira_cleaned.csv')

# Extract position vectors as a (N, 3) numpy array
observer_positions = df[['x_au', 'y_au', 'z_au']].values
julian_dates = df['jd_tdb'].values

for jd, pos in zip(julian_dates, observer_positions):
    # pos is an array-like [x, y, z]
    observer_vec = pos

bands = kete.neos.BANDS

observer = [0, 1, 0]
observer_distances = np.linalg.norm(observer_positions, axis=1)

distances = np.linspace(0.1, 6, 1000)

target_observer_vectors = target_positions - observer_positions

# Evaluate NEATM at each geometry
outputs = [
    kete.flux.neatm_model_flux(
        [r, 0, 0], observer,
        band_albedos=[0.1] * len(bands),
        diameter=939.4,
        vis_albedo=0.1,
        g_param=0.12,
        band_wavelengths=list(bands),
    )
    for r in distances
]


plt.figure(figsize=(6, 5), dpi=150)
plt.subplot(2, 1, 1)
for band_idx in range(len(bands)):
    therm_flux = [o.thermal_fluxes[band_idx] for o in outputs]
    total_flux = [o.fluxes[band_idx] for o in outputs]

    ratio = np.array(therm_flux) / total_flux
    plt.plot(distances, ratio, label=f"Band {band_idx + 1}")

plt.ylabel("Ratio Thermal / Total flux")
plt.legend()

plt.subplot(2, 1, 2)
fluxes = np.array([o.fluxes for o in outputs]).T

plt.plot(distances, fluxes[1] / fluxes[0], c="C2")

plt.xlabel("Heliocentric distance (au)")
plt.ylabel("Flux Ratio (Band 2 / Band 1)")
plt.tight_layout()
plt.close()


FileNotFoundError: [Errno 2] No such file or directory: 'atira_cleaned.csv'

In [ ]:
import glob
import os

cleaned_files = glob.glob('cleaned_data/*_cleaned.csv')

dataset_map = {}
for file in cleaned_files:
    object_type = os.path.basename(file).replace('_cleaned.csv', '')
    dataset_map[object_type] = pd.read_csv(file)

# Run batch processing pipeline
for object_type, data in dataset_map.items():
    # Execute batch calculations across each catalog
    pass

In [ ]:
np.random.seed(42)

# True physical parameters
# km
true_diam = 10.0
# geometric albedo
true_vis_albedo = 0.15
# NEATM beaming parameter
true_beaming = 1.2
# HG phase parameter
true_g_param = 0.15
# thermal emissivity (fixed, not fitted)
true_emissivity = 0.9
# IR-to-visible albedo ratio
true_r_ir = 1.6

# Derive H magnitude from diameter and albedo
true_h_mag = kete.compute_h_mag(true_diam, true_vis_albedo)
print(f"True parameters:")
print(f"  Diameter:  {true_diam} km")
print(f"  Albedo:    {true_vis_albedo}")
print(f"  H mag:     {true_h_mag:.2f}")
print(f"  Beaming:   {true_beaming}")
print(f"  G param:   {true_g_param}")
print(f"  R_IR:      {true_r_ir}")

In [ ]:
# Main-belt orbit
epoch = kete.Time.from_ymd(2024, 3, 15)
elements = kete.CometElements(
    desig="TestAsteroid",
    epoch=epoch,
    eccentricity=0.08,
    inclination=10.0,
    peri_arg=75.0,
    lon_of_ascending=45.0,
    peri_time=kete.Time(epoch.jd - 200),
    peri_dist=2.5,
)
state = elements.state
print(f"Orbit: a={elements.semi_major:.3f} AU, "
      f"e={elements.eccentricity:.3f}, "
      f"i={elements.inclination:.1f} deg")

In [ ]:
obs_jds = epoch.jd + np.array([0, 1, 5, 6, 14, 15, 29, 30])
print(f"Observing over {obs_jds[-1] - obs_jds[0]:.0f} day arc")

geometries = []
for jd in obs_jds:
    obj_state = kete.propagate_two_body(state, jd)
    earth = kete.spice.get_state("Earth", jd)
    sun2obj = obj_state.pos
    sun2obs = earth.pos
    geometries.append((sun2obj, sun2obs))

# Print example geometry
sun2obj_0, sun2obs_0 = geometries[0]
helio_dist = sun2obj_0.r
geo_dist = (sun2obj_0 - sun2obs_0).r
print(f"First epoch:  r_helio={helio_dist:.3f} AU, "
      f"r_geo={geo_dist:.3f} AU")

In [ ]:
ir_albedo = true_r_ir * true_vis_albedo
band_names = ["W1", "W2", "W3", "W4"]

# Evaluate the model at each geometry
model_outputs = [
    kete.flux.neatm_model_flux(
        sun2obj, sun2obs,
        band_albedos=[ir_albedo] * 4,
        h_mag=true_h_mag,
        vis_albedo=true_vis_albedo,
        diameter=true_diam,
        beaming=true_beaming,
        g_param=true_g_param,
        emissivity=true_emissivity,
        bands="wise",
    )
    for sun2obj, sun2obs in geometries
]

# Extract true fluxes for all four bands
true_fluxes = {}
for idx, name in enumerate(band_names):
    true_fluxes[name] = [out.fluxes[idx] for out in model_outputs]

for name in band_names:
    print(f"True {name} fluxes (Jy): "
          f"{[f'{f:.4f}' for f in true_fluxes[name]]}")

In [ ]:
# 5% flux uncertainty
sigma_frac = 0.05
observations = []

for i, (sun2obj, sun2obs) in enumerate(geometries):
    for name in band_names:
        true_flux = true_fluxes[name][i]
        sigma = sigma_frac * true_flux
        noisy_flux = true_flux + np.random.normal(0, sigma)
        obs = kete.flux.FluxObs(
            flux=noisy_flux,
            sigma=sigma,
            band=name,
            sun2obj=sun2obj,
            sun2obs=sun2obs,
        )
        observations.append(obs)

n_bands = len(band_names)
print(f"Created {len(observations)} FluxObs "
      f"({len(observations) // n_bands} epochs x {n_bands} bands)")

In [ ]:
result = kete.flux.fit_model(
    model="neatm",
    obs=observations,
    h_mag=true_h_mag,
    g_param=true_g_param,
    emissivity=true_emissivity,
    num_chains=4,
    num_tune=200,
    num_draws=500,
)

print(f"MCMC complete: {len(result.draws)} draws, "
      f"{result.n_divergent} divergent")
print(f"reduced chi2 = {result.chi2_best:.2f} "
      f"(nobs = {result.nobs}, dof = {result.nobs - 6})")
print(f"\nPosterior summary:")
print(f"  Diameter:  {result.diameter}")
print(f"  Albedo:    {result.vis_albedo}")
print(f"  Beaming:   {result.beaming}")
print(f"  H mag:     {result.h_mag}")
print(f"  G param:   {result.g_param}")
print(f"  R_IR:      {result.ir_albedo_ratio}")

In [ ]:
draws = np.array(result.draws)
columns = result.columns
true_vals = {
    "diameter": true_diam,
    "vis_albedo": true_vis_albedo,
    "beaming": true_beaming,
    "h_mag": true_h_mag,
    "g_param": true_g_param,
    "ir_albedo_ratio": true_r_ir,
}
labels = {
    "diameter": "Diameter (km)",
    "vis_albedo": r"Geometric Albedo $p_V$",
    "beaming": "Beaming",
    "h_mag": "H magnitude",
    "g_param": "G parameter",
    "ir_albedo_ratio": r"$R_{IR}$",
}

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
fig.suptitle("NEATM Posterior -- Simulated WISE W1-W4")

for ax, col_name in zip(axes.flat, labels):
    idx = columns.index(col_name)
    vals = draws[:, idx]
    ax.hist(vals, bins=30, density=True, alpha=0.5, color="steelblue")
    ax.axvline(true_vals[col_name], color="red", ls="--",
               lw=1.5, label="Truth")
    ax.set_xlabel(labels[col_name])
    ax.legend(fontsize=8)

plt.tight_layout()
plt.close()

In [ ]:
d_idx = columns.index("diameter")
pv_idx = columns.index("vis_albedo")

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(draws[:, d_idx], draws[:, pv_idx],
           s=2, alpha=0.3, color="steelblue")
ax.scatter(true_diam, true_vis_albedo,
           c="red", s=80, marker="*", zorder=5, label="Truth")
ax.set_xlabel("Diameter (km)")
ax.set_ylabel(r"Geometric Albedo $p_V$")
ax.set_title(r"Diameter vs. Albedo ($D^2 \cdot p_V$ degeneracy)")
ax.legend()
plt.tight_layout()
plt.close()

In [ ]:
residuals = np.array(result.best_fit_residuals)
fit_fluxes = np.array(result.best_fit_fluxes)
obs_fluxes = np.array([o.flux for o in observations])

# Group observation indices by band
band_idx = {name: list(range(k, len(observations), n_bands))
            for k, name in enumerate(band_names)}
markers = {"W1": "D", "W2": "^", "W3": "o", "W4": "s"}
colors = {"W1": "C0", "W2": "C1", "W3": "C2", "W4": "C3"}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
for name in band_names:
    idx = band_idx[name]
    ax.errorbar(range(len(idx)), obs_fluxes[idx],
                yerr=[observations[i].sigma for i in idx],
                fmt=markers[name], ms=4, label=f"{name} obs",
                color=colors[name])
    ax.plot(range(len(idx)), fit_fluxes[idx], "x",
            ms=6, color=colors[name], label=f"{name} model")
ax.set_xlabel("Epoch index")
ax.set_ylabel("Flux (Jy)")
ax.set_title("Observed vs. MAP Model Fluxes")
ax.legend(fontsize=7, ncol=2)

ax = axes[1]
ax.bar(range(len(residuals)), residuals, color="steelblue", alpha=0.6)
ax.axhline(0, color="black", lw=0.5)
ax.axhline(1, color="gray", ls="--", lw=0.5)
ax.axhline(-1, color="gray", ls="--", lw=0.5)
ax.set_xlabel("Observation index")
ax.set_ylabel("Normalized residual")
ax.set_title(f"MAP Residuals ($\\chi^2_\\nu$ = {result.chi2_best:.2f}, "
             f"nobs = {result.nobs})")
plt.tight_layout()
plt.close()

In [ ]:
custom_priors = kete.flux.FluxPriors(
    beaming=kete.flux.ParamPrior(
        bounds=(0.5, 3.0),
        # tighter sigma
        gaussian=(1.0, 0.1),
    ),
)
result_custom = kete.flux.fit_model(
    model="neatm",
    obs=observations,
    h_mag=true_h_mag,
    g_param=true_g_param,
    priors=custom_priors,
    num_chains=4,
    num_tune=200,
    num_draws=500,
)
print(f"Default beaming: {result.beaming}")
print(f"Custom  beaming: {result_custom.beaming}")

In [25]:
result_frm = kete.flux.fit_model(
    model="frm",
    obs=observations,
    h_mag=true_h_mag,
    g_param=true_g_param,
    emissivity=true_emissivity,
    num_chains=4,
    num_tune=200,
    num_draws=500,
)

print("FRM posterior summary:")
print(f"  Diameter:  {result_frm.diameter}")
print(f"  Albedo:    {result_frm.vis_albedo}")
print(f"  H mag:     {result_frm.h_mag}")
print(f"  R_IR:      {result_frm.ir_albedo_ratio}")
print(f"  reduced chi2: {result_frm.chi2_best:.2f} "
      f"(nobs = {result_frm.nobs}, dof = {result_frm.nobs - 5})")
print(f"\nNEATM reduced chi2: {result.chi2_best:.2f}")
print(f"FRM  reduced chi2:  {result_frm.chi2_best:.2f}")

FRM posterior summary:
  Diameter:  Data(median=14.2137, std=0.4582, ci=[13.2936, 15.0918], n=2000)
  Albedo:    Data(median=0.0802, std=0.0118, ci=[0.0617, 0.1092], n=2000)
  H mag:     Data(median=12.5938, std=0.1400, ci=[12.2987, 12.8430], n=2000)
  R_IR:      Data(median=1.6740, std=0.1929, ci=[1.2771, 1.9821], n=2000)
  reduced chi2: 1.39 (nobs = 32, dof = 27)

NEATM reduced chi2: 1.72
FRM  reduced chi2:  1.39


In [26]:
draws_frm = np.array(result_frm.draws)
columns_frm = result_frm.columns

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, col_name, label, true_val in [
    (axes[0], "diameter", "Diameter (km)", true_diam),
    (axes[1], "vis_albedo", r"Geometric Albedo $p_V$", true_vis_albedo),
]:
    neatm_vals = draws[:, columns.index(col_name)]
    frm_vals = draws_frm[:, columns_frm.index(col_name)]
    ax.hist(neatm_vals, bins=30, density=True, alpha=0.5,
            color="steelblue", label="NEATM")
    ax.hist(frm_vals, bins=30, density=True, alpha=0.5,
            color="darkorange", label="FRM")
    ax.axvline(true_val, color="red", ls="--", lw=1.5, label="Truth")
    ax.set_xlabel(label)
    ax.legend(fontsize=8)

fig.suptitle("NEATM vs. FRM Posterior Comparison")
plt.tight_layout()
plt.close()

In [ ]:
residuals_frm = np.array(result_frm.best_fit_residuals)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

ax = axes[0]
ax.bar(range(len(residuals)), residuals, color="steelblue", alpha=0.6)
ax.axhline(0, color="black", lw=0.5)
ax.axhline(1, color="gray", ls="--", lw=0.5)
ax.axhline(-1, color="gray", ls="--", lw=0.5)
ax.set_xlabel("Observation index")
ax.set_ylabel("Normalized residual")
ax.set_title(f"NEATM ($\\chi^2_\\nu$ = {result.chi2_best:.2f})")

ax = axes[1]
ax.bar(range(len(residuals_frm)), residuals_frm, color="darkorange", alpha=0.6)
ax.axhline(0, color="black", lw=0.5)
ax.axhline(1, color="gray", ls="--", lw=0.5)
ax.axhline(-1, color="gray", ls="--", lw=0.5)
ax.set_xlabel("Observation index")
ax.set_title(f"FRM ($\\chi^2_\\nu$ = {result_frm.chi2_best:.2f})")

fig.suptitle("Residual Comparison: NEATM vs. FRM")
plt.tight_layout()
plt.close()